# Chapter 2. Vector Similarity Search and Hybrid Search

Creating a knowledge graph can be an iterative process where we start with unstructured data and then add structure to it. 

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

## Components of a RAG Architecture

In a RAG application, there are two main components:
- a *retriever* that finds relevant information, and
- a *generator* that uses that information to create a response.

### The Retriever

The goal of the retriever is to find relevant information and pass that information to the generator.

**Vector Index**

A vector index is a data structure that stores vectors in a way that makes it easy to search for similar vectors by using an **approximate nearest neighbor search** algorithm. This allows the retriever to quickly find relevant information based on the similarity of vector representations.

**Vector Similarity Search Function**

A vector similarity search function is a function that takes a vector as input and returns a list of similar vectors. Two common similarity functions are:
- *Euclidean distance*, which represents the content and intensity of the text, and
- *Cosine similarity*, which measures the angle between two vectors, representing the semantic similarity of the text.

**Embedding Model**

An embedding is a way to represent complex data as a set of numbers in a simpler, lower-dimensional space.


An embedding model provide a uniform way to represent different types of data.

**Text Chunking**

Text chunking is the process of breaking up text into smaller pieces to improve the accuracy of the retriever. The presence of smaller pieces of text means that the embedding is narrower and more specific; thus the retriever will find more relevant information when searching.

**Retriever Pipeline**

The retriever pipeline takes a query as input, converts it into an embedding using the embedding model, and then uses the vector similarity search function to find similar embeddings.

### The Generator

The generator uses the information retrieved by the retriever to create a response. The generator can be a language model that takes the retrieved information as context and generates a response based on that context.

The generator does not need to know everything but it needs to know how to use the information provided by the retriever to create a coherent and relevant response. This is a much smaller task than trying to generate a response from scratch without any context, so we can use a smaller language model for the generator, which is more efficient and cost-effective.

## RAG Using Vector Similarity Search   

The figure below shows the data flow for the RAG application.

![Data flow for RAG application](./imgs/rag-data-flow.png)

We will separate the application into two stages:
- Data setup
- Query time

### Application Data Setup

The data will be stored in text chunks in a database, and the vector index will be poplated with the embeddings of the text chunks.

At run time, when a user asks a question, the question will be embedded using the same embedding model as text chunks, and then the vector index will be used to find similar text chunks.

### Text Chunking

For the paper "Einstein's Patents and Inventions", we will use a sliding window with a size of 500 characters and an overlap of 40 characters to create text chunks.

To help the embedding model better classify the semantics of each chunk, we will only chunk at spaces, so we don’t have broken words at the start and end of each chunk.

In [2]:
import requests

remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "ch02-downloaded.pdf"

response = requests.get(remote_pdf_url)

if response.status_code == 200:
    with open(pdf_filename, 'wb') as f:
        f.write(response.content)
else:
    print(f"Failed to download the PDF. Status code: {response.status_code}")

In [3]:
import pdfplumber

text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

print(text[:500])  # Print the first 500 characters of the extracted text

Einstein’s Patents and Inventions
Asis Kumar Chaudhuri
Variable Energy Cyclotron Centre
1‐AF Bidhan Nagar, Kolkata‐700 064
Abstract: Times magazine selected Albert Einstein, the German born Jewish Scientist as the person of the 20th
century. Undoubtedly, 20th century was the age of science and Einstein’s contributions in unravelling mysteries
of nature was unparalleled. However, few are aware that Einstein was also a great inventor. He and his
collaborators had patented a wide variety of inventi


In [4]:
from utils.utils import chunk_text

chunks = chunk_text(text, chunk_size=500, overlap=40)
print(f"Total number of chunks: {len(chunks)}")
print(f"First chunk:\n{chunks[0]}")

Total number of chunks: 89
First chunk:
Einstein’s Patents and Inventions
Asis Kumar Chaudhuri
Variable Energy Cyclotron Centre
1‐AF Bidhan Nagar, Kolkata‐700 064
Abstract: Times magazine selected Albert Einstein, the German born Jewish Scientist as the person of the 20th
century. Undoubtedly, 20th century was the age of science and Einstein’s contributions in unravelling mysteries
of nature was unparalleled. However, few are aware that Einstein was also a great inventor. He and his
collaborators had patented a wide variety of inventions


### Embedding Model

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

True

In [6]:
openai_client = OpenAI()

In [7]:
def embed(texts):
    response = openai_client.embeddings.create(
        input=texts,
        model='text-embedding-3-small',
    )

    return list(map(lambda x: x.embedding, response.data))

In [9]:
embeddings = embed(chunks)

print(f"Number of embeddings generated: {len(embeddings)}")
print(f"Embedding for the first chunk:\n{embeddings[0][0:3]}...")

Number of embeddings generated: 89
Embedding for the first chunk:
[0.025176985189318657, -0.023162826895713806, -0.013964015990495682]...


### Database with Vectoro Similarity Search Function

Neo4j has a built-in vector index and it is easy to use.

The data model is simple. We will have a single node type `Chunk` with two properties: `text` and `embedding`:
- `text` property holds the text of the chunk, and
- `embedding` property holds the vector representation of the chunk.

In [13]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "neo4j://127.0.0.1:7687",
    auth=("neo4j", "password")
)

Next we will create the vector index:

In [14]:
driver.execute_query("""CREATE VECTOR INDEX pdf IF NOT EXISTS
FOR (c:Chunk)
ON c.embedding""")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x0000023DD5B8C910>, keys=[])

We will name the vector index `pdf` and it will be used to index nodes of type `Chunk` on the `embedding` property using the cosine similarity function.

Now that we have a vector index, we can populate it with the embeddings.

We will do this using Cypher, where we first create a node for each chunk and then set the `text` and `embedding` properties of the node. We will also store an index on each `:Chunk` node so that we can easily find the chunk later:

In [16]:
cypher_query = """
WITH $chunks as chunks, range(0, size($chunks)) AS index
UNWIND index AS i
WITH i, chunks[i] AS chunk, $embeddings[i] AS embedding
MERGE (c:Chunk {index: i})
SET c.text = chunk, c.embedding = embedding
"""

driver.execute_query(
    cypher_query,
    chunks=chunks,
    embeddings=embeddings
)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x0000023DD5B95220>, keys=[])

Explore in Neo4j Browser:

![Neo4j Browser](./imgs/store-chunks-and-populate-vector-index.png)

To check what is in the database:

In [17]:
records, _, _ = driver.execute_query("MATCH (c:Chunk) WHERE c.index = 0 RETURN c.embedding, c.text")

print(records[0]["c.text"][0:30])
print(records[0]["c.embedding"][0:3])

Einstein’s Patents and Inventi
[0.025176985189318657, -0.023162826895713806, -0.013964015990495682]


### Performing Vector Search

To perform a vector similarity search, we first need to embed the question that we want to answer.

In [18]:
question = "At what time was Einstein really interested in experimental works?"
# Embed the question
question_embedding = embed([question])[0]

Now we can perform a vector similarity search using Cypher:

In [19]:
query = """
CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node AS hits, score
RETURN hits.text AS text, score, hits.index AS index
"""

similar_records, _, _ = driver.execute_query(
    query,
    k=5,
    question_embedding=question_embedding
)

In [21]:
for record in similar_records:
    print(record['text'])
    print(f"Score: {record['score']} | Index: {record['index']}")
    print('=====')

CH‐Switzerland
Considering Einstein’s upbringing, his interest in inventions and patents was not unusual.
Being a manufacturer’s son, Einstein grew upon in an environment of machines and instruments.
When his father’s company obtained the contract to illuminate Munich city during beer festival, he
was actively engaged in execution of the contract. In his ETH days Einstein was genuinely interested
in experimental works. He wrote to his friend, “most of the time I worked in the physical laboratory,
fascinated by the direct contact with observation.” Einstein's
Score: 0.8110630512237549 | Index: 42
=====
Einstein
left his job at the Patent office and joined the University of Zurich on October 15, 1909. Thereafter, he
continued to rise in ladder. In 1911, he moved to Prague University as a full professor, a year later, he
was appointed as full professor at ETH, Zurich, his alma‐mater. In 1914, he was appointed Director of
the Kaiser Wilhelm Institute for Physics (1914–1932) and a professor

### Generating an Answer Using an LLM

In [23]:
sys_msg = "You're en Einstein expert, but can only use the provided documents to respond to the questions."
user_msg = f"""
Use the following documents to answer the question that will follow:
{[doc["text"] for doc in similar_records]}

---

The question to answer using information only from the above documents: {question}
"""

print("Question: ", question)

stream = openai_client.chat.completions.create(
    model='gpt-4.1',
    messages=[
        {'role': 'system', 'content': sys_msg},
        {'role': 'user', 'content': user_msg}
    ],
    stream=True
)

for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")

Question:  At what time was Einstein really interested in experimental works?
Based on the provided documents, Einstein was genuinely interested in experimental works during his ETH days (his time as a student at the Swiss Federal Polytechnic in Zurich). The relevant excerpt states:

"In his ETH days Einstein was genuinely interested in experimental works. He wrote to his friend, 'most of the time I worked in the physical laboratory, fascinated by the direct contact with observation.'"

## Adding Full-Text Search to the RAG Application to Enable Hybrid Search

While pure vector similarity search can take us a long way and is a great improvement over plain full-text search, it is often not enough to produce high enough quality, accuracy, and performance for production use cases.

In this section, we will consider how to add *full-text search* to the RAG application to enable *hybrid search*.

### Full-Text Search Index

**Full-text search** is a text search method in databases that searches for matches in the data via keywords and not by similarity in a vector space. The search term must be an exact match to the text in the database.

To enable hybrid search, we need to add a full-text search index to the database in Neo4j.

In [24]:
try:
    driver.execute_query(f"CREATE FULLTEXT INDEX ftPdfChunk FOR (c:Chunk) ON EACH [c.text]")
except:
    print("Full-text search index already exists.")

we created a full-text search index named `ftPdfChunk` on the `text` property of the `:Chunk` nodes.

### Performing Hybrid Search

The idea of hybrid search is that we perform a vector similarity search and a full-text search and then combine the results. We need to normalize the scores to compare the two different matches. We will do this by dividing the scores by the *highest score* for each search.

In [26]:
hybrid_query = """
CALL {
    // vector index
    CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node, score
    WITH collect({node: node, score: score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    // We use 0 as min
    RETURN n.node AS node, (n.score / max) AS score
    UNION

    // full-text search
    CALL db.index.fulltext.queryNodes('ftPdfChunk', $question, {limit: $k}) YIELD node, score
    WITH collect({node: node, score: score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    // We use 0 as min
    RETURN n.node AS node, (n.score / max) AS score
}
// deduplicate results
WITH node, max(score) AS score ORDER BY score DESC LIMIT $k
RETURN node, score
"""

similar_hybrid_records, _, _ = driver.execute_query(
    hybrid_query,
    k=5,
    question_embedding=question_embedding,
    question=question
)

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL {\n    // vector index\n    CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node, score\n    WITH collect({node: node, score: score}) AS nodes, max(score) AS max\n    UNWIND nodes AS n\n    // We use 0 as min\n    RETURN n.node AS node, (n.score / max) AS score\n    UNION\n\n    // full-text search\n    CALL db.index

We used a union Cypher query where we first perform a vector similarity search and then a full-text search, and then we deduplicate the results and return the top 5 results.

In [27]:
for record in similar_hybrid_records:
    print(record['node']['text'])
    print(f"Score: {record['score']} | Index: {record['node']['index']}")
    print('=====')

CH‐Switzerland
Considering Einstein’s upbringing, his interest in inventions and patents was not unusual.
Being a manufacturer’s son, Einstein grew upon in an environment of machines and instruments.
When his father’s company obtained the contract to illuminate Munich city during beer festival, he
was actively engaged in execution of the contract. In his ETH days Einstein was genuinely interested
in experimental works. He wrote to his friend, “most of the time I worked in the physical laboratory,
fascinated by the direct contact with observation.” Einstein's
Score: 1.0 | Index: 42
=====
Einstein
left his job at the Patent office and joined the University of Zurich on October 15, 1909. Thereafter, he
continued to rise in ladder. In 1911, he moved to Prague University as a full professor, a year later, he
was appointed as full professor at ETH, Zurich, his alma‐mater. In 1914, he was appointed Director of
the Kaiser Wilhelm Institute for Physics (1914–1932) and a professor at the Humbold

In [28]:
user_msg = f"""
Use the following documents to answer the question that will follow:
{[doc["node"]["text"] for doc in similar_hybrid_records]}

---

The question to answer using information only from the above documents: {question}
"""

print("Question:", question)

stream = openai_client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_msg}
    ],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")

Question: At what time was Einstein really interested in experimental works?
Einstein was genuinely interested in experimental works during his ETH days.